# 04 · El motor GMM: validación con el IASB y vida temporal

`gmm.run_period` mide un grupo durante un periodo. Recibe valores presentes que calcula cada producto y aplica la norma:

- fija el CSM de los contratos nuevos (par. 38) o la pérdida si son onerosos (par. 47);
- acrecienta el CSM a tasa bloqueada (B72(b));
- lo ajusta por cambios relacionados con servicio futuro medidos a tasa bloqueada (B96, B72(c));
- lo libera con unidades de cobertura (B119);
- lleva el componente de pérdida (par. 49–52).

Antes de medir a Seguros Barú, el motor reproduce los Ejemplos Ilustrativos del IASB.

In [1]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
pd.set_option('display.float_format', lambda x: f'{x:,.0f}' if abs(x) >= 100 else f'{x:,.4f}')
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
from ifrs17.gmm import GroupState, PeriodInputs, run_period
I = 0.05
pv = lambda flujos: sum(cf / (1 + I) ** (t + 1) for t, cf in enumerate(flujos))
a = run_period(GroupState(), PeriodInputs(nb_pv_inflows=900, nb_pv_outflows=pv([200] * 3), nb_ra=120))
b = run_period(GroupState(), PeriodInputs(nb_pv_inflows=900, nb_pv_outflows=pv([400] * 3), nb_ra=120))
pd.DataFrame({'Ejemplo 1A': a.movements['new_contracts'], 'Ejemplo 1B': b.movements['new_contracts']})

,Ejemplo 1A,Ejemplo 1B
pv,-355,189
ra,120,120
csm,235,0.0000
loss,0.0000,309


El Ejemplo 1A da un CSM de 235 (prima 900 menos valor presente 545 menos ajuste por riesgo 120) y el 1B una pérdida de 309, igual que el IASB. Las pruebas `tests/test_gmm_iasb_examples.py` cubren también los Ejemplos 2A, 2B, 7, 8 y 12C. Abajo se reproduce el 2B, donde el grupo se vuelve oneroso en el año 2.

In [2]:
import importlib.util
spec = importlib.util.spec_from_file_location('ie', '../tests/test_gmm_iasb_examples.py'); ie = importlib.util.module_from_spec(spec); spec.loader.exec_module(ie)
ra = {'old': [80, 40, 0], 'new': [80, 88, 0]}
y1, y2, y3 = ie.three_years([200, 200, 200], ra, changes={('actual', 1): 400, ('estimate', 1): [450]})
pd.DataFrame(y2.movements).T[['pv', 'ra', 'csm']]

,pv,ra,csm
new_contracts,0.0000,0.0000,-0.0000
future_service_csm,144,29.0224,-173
future_service_onerous,94.1351,18.9776,0.0000
csm_release,0.0000,0.0000,-0.0000
ra_release,0.0000,-40.0000,0.0000
experience,200,0.0000,0.0000
finance,18.5941,0.0000,8.2373
premiums_received,0.0000,0.0000,0.0000
claims_and_expenses_incurred,-400,0.0000,0.0000
investment_components,-0.0000,0.0000,0.0000


El aumento de flujos relacionado con servicio futuro (238 de valor presente y 48 de ajuste por riesgo) agota el CSM de 173 y deja una pérdida de 113 (par. 48), como en el párrafo IE26.

**Una diferencia documentada.** En el Ejemplo 8 el IASB reconoce en el año 3 todo el CSM que nace de la reversión de pérdidas al cierre del año 2. El motor, en cambio, reparte el CSM del final del periodo, después de todos los ajustes, entre las unidades del periodo y las futuras (par. 44(e), B119), igual que en el Ejemplo 2A. La prueba reproduce el Ejemplo 8 cuando al año 2 no se le asignan unidades.

## Vida temporal en los tres cierres

En cada cierre, cada grupo se proyecta póliza por póliza con la mejor estimación nueva y se arma el análisis del cambio:

- **Futuro esperado:** la proyección del cierre anterior se acrecienta a tasa bloqueada.
- **Servicio futuro (CSM):** la diferencia con la medición nueva a tasa bloqueada.
- **Servicio corriente:** la experiencia de siniestros y gastos del periodo.
- **Efecto de tasas:** la diferencia entre la tasa corriente y la bloqueada va a ORI (par. 88(b)).

In [3]:
from ifrs17 import closing as C
res = C.run()
filas = []
for o in res['vt']['closings']:
    for g, r in o.results.items():
        filas.append({'cierre': o.closing, 'grupo': g, 'CSM': r.closing.csm, 'componente_pérdida': r.closing.loss_component,
                      'pérdida_inicial': r.movements['new_contracts']['loss'], 'ajuste_CSM_servicio_futuro': r.movements['future_service_csm']['csm'],
                      'pérdidas_y_reversiones': r.pnl['ise_onerous_losses_and_reversals'], 'ingreso': r.pnl['insurance_revenue'],
                      'resultado_servicio': r.pnl['insurance_service_result'], 'finanzas_resultados': r.pnl['ifie_pnl'], 'finanzas_ORI': r.pnl['ifie_oci']})
pd.DataFrame(filas)

,cierre,grupo,CSM,componente_pérdida,pérdida_inicial,ajuste_CSM_servicio_futuro,pérdidas_y_reversiones,ingreso,resultado_servicio,finanzas_resultados,finanzas_ORI
0,2023-12-31,VT-2023-ONE,0.0000,"217,915","163,680",-0.0000,"217,844","151,074","-274,388","1,325","13,261"
1,2023-12-31,VT-2023-RES,"492,964",0.0000,0.0000,"-78,521",0.0000,"315,269","-79,871","-2,289","11,499"
2,2023-12-31,VT-2023-SIG,"2,233,051",0.0000,0.0000,"-259,695",0.0000,"722,783","189,790","-9,708",741
3,2024-12-31,VT-2023-ONE,0.0000,"379,342",0.0000,-0.0000,"160,958","294,449","57,294","6,623","-21,508"
4,2024-12-31,VT-2023-RES,0.0000,"172,455",0.0000,"-518,908","172,455","566,586","113,568",-41.6938,"-21,638"
5,2024-12-31,VT-2023-SIG,"2,232,065",0.0000,0.0000,"113,091",0.0000,"1,540,544","934,510","-11,752","-28,312"
6,2024-12-31,VT-2024-ONE,0.0000,"178,158","151,336",-0.0000,"178,107","131,168","-83,101",917,"-9,236"
7,2024-12-31,VT-2024-RES,0.0000,"134,152",0.0000,"-624,398","134,152","326,426","-147,839","-1,930","-19,644"
8,2024-12-31,VT-2024-SIG,"2,553,299",0.0000,0.0000,"289,866",0.0000,"806,010","-116,433","-9,686","-8,627"
9,2025-12-31,VT-2023-ONE,0.0000,"157,867",0.0000,0.0000,"-223,792","260,439","411,535","18,495","17,376"


La historia del guion sale del motor:

- **2023:** VT-2023-ONE reconoce una pérdida inicial. El primer estudio de mortalidad, con poca credibilidad propia, mueve poco el factor.
- **2024:** el estudio del reasegurador sube a 115% de CSO 1980 la mortalidad de edades 50+ con suma alta. Agota el CSM de VT-2023-RES y VT-2024-RES y crea un componente de pérdida (par. 48).
- **2025:** el nuevo estudio (85%) revierte primero el componente de pérdida y después reconstruye el CSM (par. 50(b)).

![CSM por grupo](../docs/figures/vida_temporal_csm.png)

In [4]:
from ifrs17 import disclosures as D
r101 = D.reconciliation_101(res)
vt101 = r101[(r101['portfolio'] == 'VT')].groupby(['closing', 'line'], sort=False)[['pv', 'ra', 'csm', 'total']].sum()
vt101.loc['2024-12-31']

,pv,ra,csm,total
line,,,,
opening,"-3,593,915","1,229,945","2,726,015","362,045"
new_contracts,"-4,020,735","1,231,289","2,940,783","151,336"
future_service_csm,"695,871","44,478","-740,350",-0.0000
future_service_onerous,"494,067","27,820",0.0000,"521,887"
csm_release,0.0000,0.0000,"-365,541","-365,541"
ra_release,0.0000,"-216,543",0.0000,"-216,543"
experience,"-849,137",0.0000,0.0000,"-849,137"
finance,"-349,291",0.0000,"224,457","-124,834"
premiums_received,"6,304,237",0.0000,0.0000,"6,304,237"
